# Debug ETL Job

This notebook demonstrates how to debug the ETL job using the debug module.

In [1]:
import os
import sys

# Add the project directory to the path
sys.path.append('/home/jovyan/work')

from debug.jupyter import JupyterDebugger
os.environ['DB_USER'] = 'postgres'
os.environ['DB_PASSWORD'] = 'postgres'
os.environ['CH_USER'] = 'default'
os.environ['CH_PASSWORD'] = ''

In [2]:
# Initialize the debugger with the config path
debugger = JupyterDebugger('/home/jovyan/work/sample_configs/v2.json')

# Initialize Spark with the necessary configurations for connecting to our services
spark = debugger.initialize_spark()

# Add S3/MinIO configurations
spark.conf.set("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
spark.conf.set("spark.hadoop.fs.s3a.access.key", "minioadmin")
spark.conf.set("spark.hadoop.fs.s3a.secret.key", "minioadmin")
spark.conf.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

RuntimeError: Java gateway process exited before sending its port number

In [3]:
# Load source data
source_dfs = debugger.load_source_data()

# Display the loaded source DataFrames
for name, df in source_dfs.items():
    print(f"\n=== {name} ===\n")
    df.show(5)


=== customers ===

+-----------+--------------+--------------------+--------------------+-----------------+--------+
|customer_id| customer_name|               email|             address|registration_date|  status|
+-----------+--------------+--------------------+--------------------+-----------------+--------+
|          1|    John Smith|john.smith@exampl...|123 Main St, New ...|       2022-01-15|  active|
|          2|      Jane Doe|jane.doe@example.com|456 Oak Ave, San ...|       2022-02-20|  active|
|          3|Robert Johnson|robert.j@example.com|789 Pine Rd, Chic...|       2022-03-10|  active|
|          4|  Emily Wilson| emily.w@example.com|321 Cedar Ln, Bos...|       2022-04-05|inactive|
|          5| Michael Brown|michael.b@example...|654 Maple Dr, Sea...|       2022-05-12|  active|
+-----------+--------------+--------------------+--------------------+-----------------+--------+
only showing top 5 rows


=== orders ===

+--------+-----------+----------+----------+--------+---

In [6]:
# Execute transformations
transformed_dfs = debugger.execute_transformations()

# Display the transformed DataFrames
for name, df in transformed_dfs.items():
    print(f"\n=== {name} ===\n")
    df.show(5)


=== customer_orders ===

+-----------+-------------+--------------------+--------------------+-----------------+------+--------+----------+----------+--------+-------+
|customer_id|customer_name|               email|             address|registration_date|status|order_id|product_id|order_date|quantity|  price|
+-----------+-------------+--------------------+--------------------+-----------------+------+--------+----------+----------+--------+-------+
|          1|   John Smith|john.smith@exampl...|123 Main St, New ...|       2022-01-15|active|      40|         4|2022-06-02|       4| 822.63|
|          1|   John Smith|john.smith@exampl...|123 Main St, New ...|       2022-01-15|active|      36|         2|2022-04-19|       1|1285.42|
|          1|   John Smith|john.smith@exampl...|123 Main St, New ...|       2022-01-15|active|      35|         3|2022-10-31|       1| 754.49|
|          1|   John Smith|john.smith@exampl...|123 Main St, New ...|       2022-01-15|active|      29|         4|20

In [7]:
spark.sql("select * from customer_orders").show(1)

+-----------+-------------+--------------------+--------------------+-----------------+------+--------+----------+----------+--------+------+
|customer_id|customer_name|               email|             address|registration_date|status|order_id|product_id|order_date|quantity| price|
+-----------+-------------+--------------------+--------------------+-----------------+------+--------+----------+----------+--------+------+
|          1|   John Smith|john.smith@exampl...|123 Main St, New ...|       2022-01-15|active|      40|         4|2022-06-02|       4|822.63|
+-----------+-------------+--------------------+--------------------+-----------------+------+--------+----------+----------+--------+------+
only showing top 1 row



In [8]:
# Examine the execution plan for a specific transformation
if 'customer_spending' in transformed_dfs:
    print("Execution plan for customer_spending:")
    transformed_dfs['customer_spending'].explain()

Execution plan for customer_spending:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_spent#160 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_spent#160 DESC NULLS LAST, 10), ENSURE_REQUIREMENTS, [plan_id=789]
      +- HashAggregate(keys=[customer_id#0, customer_name#1], functions=[sum(total_amount#152)])
         +- Exchange hashpartitioning(customer_id#0, customer_name#1, 10), ENSURE_REQUIREMENTS, [plan_id=786]
            +- HashAggregate(keys=[customer_id#0, customer_name#1], functions=[partial_sum(total_amount#152)])
               +- Project [customer_id#0, customer_name#1, (price#17 * cast(quantity#16L as double)) AS total_amount#152]
                  +- SortMergeJoin [product_id#14L], [cast(id#27 as bigint)], Inner
                     :- Sort [product_id#14L ASC NULLS FIRST], false, 0
                     :  +- Exchange hashpartitioning(product_id#14L, 10), ENSURE_REQUIREMENTS, [plan_id=778]
                     :     +- Project [custo

In [9]:
# Run the full job in debug mode
result = debugger.debug_job()

# Display final results
for name, df in result['transformed'].items():
    print(f"\n=== Final {name} ===\n")
    df.show(10)


=== Final customer_orders ===

+-----------+-------------+--------------------+--------------------+-----------------+------+--------+----------+----------+--------+-------+
|customer_id|customer_name|               email|             address|registration_date|status|order_id|product_id|order_date|quantity|  price|
+-----------+-------------+--------------------+--------------------+-----------------+------+--------+----------+----------+--------+-------+
|          1|   John Smith|john.smith@exampl...|123 Main St, New ...|       2022-01-15|active|      40|         4|2022-06-02|       4| 822.63|
|          1|   John Smith|john.smith@exampl...|123 Main St, New ...|       2022-01-15|active|      36|         2|2022-04-19|       1|1285.42|
|          1|   John Smith|john.smith@exampl...|123 Main St, New ...|       2022-01-15|active|      35|         3|2022-10-31|       1| 754.49|
|          1|   John Smith|john.smith@exampl...|123 Main St, New ...|       2022-01-15|active|      29|       

In [1]:
!python3 /home/jovyan/work/run.py /home/jovyan/work/sample_configs/v2.json

Listening for transport dt_socket at address: 4000
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/09 15:32:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/04/09 15:32:39 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
25/04/09 15:32:39 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
25/04/09 15:32:39 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
25/04/09 15:32:40 WARN MetricsConf

In [10]:
!du -s -h /opt/spark-data/output/order_details

du: cannot access '/opt/spark-data/output/order_details': No such file or directory
